# 08 - LSTM con TensorFlow/Keras

## Objetivo
Implementar el mismo problema de prediccion de fallas con TensorFlow/Keras
para demostrar dominio en ambos frameworks.

## Keras vs PyTorch
- Keras: API de alto nivel, menos codigo, model.fit() hace todo
- PyTorch: Mas control, training loop manual, mas flexible
- Ambos son estandar de la industria

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score

from src.data.preprocessor import FiberDataPreprocessor
from src.data.dataset import FiberSequenceDataset
from src.models.tf_lstm_predictor import create_tf_lstm_model, train_tf_lstm, save_tf_model
from src.utils.config_loader import load_config
from src.utils.visualization import apply_style, plot_training_history

apply_style()
config = load_config()

In [ ]:
# Preparar datos como numpy arrays (TF no usa DataLoader)
preprocessor = FiberDataPreprocessor()
train_df, val_df, test_df = preprocessor.process_pipeline()

feature_cols = ['optical_power_dbm', 'attenuation_db_km', 'ber', 'osnr_db',
                'chromatic_dispersion', 'temperature', 'humidity', 'log_ber']
feature_cols = [c for c in feature_cols if c in train_df.columns]

seq_len = config['preprocessing']['sequence_length']

# Crear secuencias usando nuestro Dataset de PyTorch (reutilizamos la logica)
train_ds = FiberSequenceDataset(train_df, feature_cols, 'fault_in_24h', seq_len)
val_ds = FiberSequenceDataset(val_df, feature_cols, 'fault_in_24h', seq_len)
test_ds = FiberSequenceDataset(test_df, feature_cols, 'fault_in_24h', seq_len)

X_train, y_train = train_ds.sequences, train_ds.targets
X_val, y_val = val_ds.sequences, val_ds.targets
X_test, y_test = test_ds.sequences, test_ds.targets

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}, y_val: {y_val.shape}')
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')

In [ ]:
# Crear y entrenar modelo
model = create_tf_lstm_model(
    n_features=len(feature_cols),
    sequence_length=seq_len,
)

history = train_tf_lstm(model, X_train, y_train, X_val, y_val)

In [ ]:
# Curvas de entrenamiento
fig = plot_training_history(history['loss'], history['val_loss'])
plt.show()

if 'auc' in history:
    fig = plot_training_history(history['auc'], history['val_auc'], metric_name='AUC')
    plt.show()

In [ ]:
# Evaluacion
test_results = model.evaluate(X_test, y_test, verbose=1)
print(f'\nTest Loss: {test_results[0]:.4f}')
print(f'Test Accuracy: {test_results[1]:.4f}')

y_pred_proba = model.predict(X_test).flatten()
y_pred_binary = (y_pred_proba > 0.5).astype(int)

print('\nClassification Report:')
print(classification_report(y_test, y_pred_binary, target_names=['Normal', 'Falla 24h']))
print(f'ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}')

# Guardar
save_tf_model(model)